In [1]:
import os
import json
from dotenv import load_dotenv
from llama_index.llms.groq import Groq
from llama_index.core.tools import FunctionTool
from llama_index.core import Settings

# loading Api keys
load_dotenv()
groq_api_key = os.getenv('GROQ_API_KEY')
print('Environment ready...!')


Environment ready...!


In [4]:
# llm initialization
Settings.llm = Groq(
    model= 'llama-3.3-70B-versatile',
    api_key= groq_api_key
)

# Quick test to ensure connectivity
response = Settings.llm.complete('Hello, can you hear me?')
print(f"LLM Status: {response}")

LLM Status: I can understand and respond to your text. How can I assist you today?


In [ ]:
# the python Logic(The 'Action')
# before the Schema, we need the actual function. For today, we will build a function that calculates the profit margin


def calculate_profit_margin(revenue: float, cost: float) -> str:
    '''it calculates the net profit margin percentage based on revenue and cost.'''
    if revenue <= 0:
        return 'Error: Revenue must be greater than 0.'
    
    profit = revenue - cost
    margin = (profit / revenue) * 100
    return f"The net profit margin is {round(margin, 2)}%"

print('Logic function successfully defined')

Logic function successfully defined


In [6]:
# manual JSON schema 
# we must write this manually to understand how the LLM 'sees' our function
# creating the JSON definition manually

manual_tools_spec = {
    "name": "calculate_profit_margin",
    "description": "Calculate the net profit margin percentage based on revenue and cost.",
    "parameters": {
        "type": "object",
        "properties": {
            "revenue": "number",
            "description": "The total gross revenue earned."
        },
        "cost": {
            "type": "number",
            "description": "The total expenses/costs incurred."
        },
        "required": ["revenue", "cost"]
    }
}

print("Manual Schema Constructed")
print(json.dumps(manual_tools_spec, indent=2))

Manual Schema Constructed
{
  "name": "calculate_profit_margin",
  "description": "Calculate the net profit margin percentage based on revenue and cost.",
  "parameters": {
    "type": "object",
    "properties": {
      "revenue": "number",
      "description": "The total gross revenue earned."
    },
    "cost": {
      "type": "number",
      "description": "The total expenses/costs incurred."
    },
    "required": [
      "revenue",
      "cost"
    ]
  }
}


In [ ]:
# automation with llama-index 
# now we use the library to do the same thing. This verifies your manual work against a the production-standard generator


# llamaindex automatically generates the spec from the docstrings and type hints
margin_tool = FunctionTool.from_defaults(fn = calculate_profit_margin)

# Verify the auto-generated schema
auto_spec = margin_tool.metadata.get_parameters_dict()
print("Auto-Generated Schema:")
print(json.dumps(auto_spec, indent = 2))

Auto-Generated Schema:
{
  "properties": {
    "revenue": {
      "title": "Revenue",
      "type": "number"
    },
    "cost": {
      "title": "Cost",
      "type": "number"
    }
  },
  "required": [
    "revenue",
    "cost"
  ],
  "type": "object"
}


In [8]:
# automation with llama-index
# this is the moment of truth where the llm will decide to use my tool instead
# of just guessing the answer

# asking a question that requires the tool
query = "My company made $500,000 in revenue last year with $320,000 in costs. What was our profit margin?"

# we use the LLM to 'predict' the tool call
response = Settings.llm.predict_and_call(
    [margin_tool],
    user_msg = query
)

print(f"Final Answer: {response}")

Final Answer: The net profit margin is 36.0%
